# Gradio Basics — A Beginner-Friendly Notebook

This notebook takes you from zero to building small AI-powered apps with **Gradio**, Python's library for turning ordinary functions into web interfaces.

## Roadmap

**Part 1 — Fundamentals:** installation, your first app, and the core input components (Textbox, Number, Slider, Dropdown, Radio, Checkbox)

**Part 2 — Inputs, Outputs & Events:** multiple inputs/outputs, buttons, click/change/submit events, file upload, images, dataframes

**Part 3 — Layout & Advanced Basics:** Blocks, Row/Column, Tabs, Markdown, State, ClearButton, Chatbot, ChatInterface

**Part 4 — AI Applications:** wiring Gradio up to OpenAI, and a look at where RAG fits in

By the end, you'll be ready to build the five mini projects listed at the bottom: a text summarizer, a sentiment analysis app, a PDF Q&A app, a study assistant, and a RAG document chatbot.

# Part 1 — Gradio Fundamentals

## 1. What is Gradio?

Gradio is a Python library that turns a plain Python function into a shareable web interface — no HTML, CSS, or JavaScript required for the basics.

Without Gradio, you might write:

```python
name = input("Enter your name: ")
print("Hello", name)
```

With Gradio, the same idea becomes a real web page:

```
Web Browser → Textbox → Python Function → Output
```

You write the function; Gradio builds and serves the page.

## 2. Installation

Install Gradio, then check the version to confirm it's working.

In [ ]:
%pip install gradio

In [ ]:
import gradio as gr

print(gr.__version__)

## 3. Your First Gradio App

The smallest possible Gradio app: one input, one output, one function.

In [ ]:
import gradio as gr

def greet(name):
    return "Hello " + name

demo = gr.Interface(
    fn=greet,
    inputs="text",
    outputs="text"
)

demo.launch()

In [ ]:
# Adding share=True means that it can be accessed publically
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# If you're at work on on a work network, I suggest skip this test.

gr.Interface(fn=greet, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=greet, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
gr.Interface(fn=greet, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("mob", "12345"))

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface


gr.Interface(fn=greet, inputs="textbox", outputs="textbox", flagging_mode="never", theme=gr.themes.Glass()).launch()

**What each piece does:**

- `gr.Interface()` — builds the app
- `fn=greet` — the Python function to run
- `inputs="text"` — creates a text input box
- `outputs="text"` — creates a text output box
- `demo.launch()` — starts the local web server

## 4. Textbox

`gr.Textbox` gives you more control than the shorthand `"text"` string — you can add labels, placeholders, and more.

In [ ]:
import gradio as gr

def welcome(name):
    return f"Welcome, {name}!"

demo = gr.Interface(
    fn=welcome,
    inputs=gr.Textbox(label="Enter your name"),
    outputs=gr.Textbox(label="Message")
)

demo.launch()

## 5. Number Input

`gr.Number` restricts input to numeric values, which is handy for anything mathematical.

In [ ]:
import gradio as gr

def square(number):
    return number ** 2

demo = gr.Interface(
    fn=square,
    inputs=gr.Number(label="Enter a number"),
    outputs=gr.Number(label="Square")
)

demo.launch()

Try entering `5` — you should get `25` back.

## 6. Slider

Sliders are great for bounded numeric ranges, like a birth year.

In [ ]:
import gradio as gr

def calculate_age(year):
    return 2026 - year

demo = gr.Interface(
    fn=calculate_age,
    inputs=gr.Slider(
        minimum=1950,
        maximum=2026,
        value=2000,
        step=1,
        label="Birth Year"
    ),
    outputs=gr.Number(label="Age")
)

demo.launch()

## 7. Dropdown

Use `gr.Dropdown` when the user should pick exactly one option from a fixed list.

In [ ]:
import gradio as gr

def show_language(language):
    return f"You selected {language}"

demo = gr.Interface(
    fn=show_language,
    inputs=gr.Dropdown(
        choices=["Python", "Java", "C++", "JavaScript"],
        label="Select Language"
    ),
    outputs="text"
)

demo.launch()

## 8. Radio Button

`gr.Radio` works like a dropdown, but shows all options at once — nicer for a small number of choices.

In [ ]:
import gradio as gr

def show_gender(gender):
    return f"You selected: {gender}"

demo = gr.Interface(
    fn=show_gender,
    inputs=gr.Radio(
        ["Male", "Female", "Other"],
        label="Select"
    ),
    outputs="text"
)

demo.launch()

## 9. Checkbox

`gr.Checkbox` passes a plain `True`/`False` into your function.

In [ ]:
import gradio as gr

def check_status(agree):
    if agree:
        return "You agreed."
    return "You did not agree."

demo = gr.Interface(
    fn=check_status,
    inputs=gr.Checkbox(label="I agree"),
    outputs="text"
)

demo.launch()

# Part 2 — Inputs, Outputs & Events

## 10. Multiple Inputs

A function can take several arguments — just pass a list of components in the same order as the function's parameters.

In [ ]:
import gradio as gr

def add_numbers(a, b):
    return a + b

demo = gr.Interface(
    fn=add_numbers,
    inputs=[
        gr.Number(label="Number 1"),
        gr.Number(label="Number 2")
    ],
    outputs=gr.Number(label="Result")
)

demo.launch()

Gradio maps each input component to a parameter in order — the first component fills `a`, the second fills `b`.

## 11. Multiple Outputs

Return a tuple from your function to fill several output components at once.

In [ ]:
import gradio as gr

def calculate(a, b):
    total = a + b
    multiplication = a * b

    return total, multiplication

demo = gr.Interface(
    fn=calculate,
    inputs=[
        gr.Number(label="Number 1"),
        gr.Number(label="Number 2")
    ],
    outputs=[
        gr.Number(label="Addition"),
        gr.Number(label="Multiplication")
    ]
)

demo.launch()

## 12. Button (with Blocks)

So far, `gr.Interface` has run our function automatically whenever an input changes. Switching to `gr.Blocks` gives finer control — for example, only running the function when a button is clicked.

In [ ]:
import gradio as gr

def greet(name):
    return f"Hello {name}!"

with gr.Blocks() as demo:

    name = gr.Textbox(label="Name")
    button = gr.Button("Greet")
    output = gr.Textbox(label="Output")

    button.click(
        fn=greet,
        inputs=name,
        outputs=output
    )

demo.launch()

`button.click(...)` means: *run this function only when the button is clicked.*

## 13. Events

Beyond `click`, Gradio components support several other events, including `change`, `input`, `submit`, `select`, and `upload`.

Here, the function fires automatically whenever the textbox's value changes.

In [ ]:
import gradio as gr

def convert(text):
    return text.upper()

with gr.Blocks() as demo:

    textbox = gr.Textbox()
    output = gr.Textbox()

    textbox.change(
        fn=convert,
        inputs=textbox,
        outputs=output
    )

demo.launch()

## 14. Submit Event

`submit` fires when the user presses **Enter** inside the textbox — useful for chat-like inputs.

In [ ]:
import gradio as gr

def process(text):
    return text.upper()

with gr.Blocks() as demo:

    input_box = gr.Textbox(label="Enter text")
    output = gr.Textbox(label="Output")

    input_box.submit(
        fn=process,
        inputs=input_box,
        outputs=output
    )

demo.launch()

## 15. Clear Button

`gr.ClearButton` resets one or more components back to their default values with a single click.

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    name = gr.Textbox()
    output = gr.Textbox()

    clear = gr.ClearButton(
        components=[name, output]
    )

demo.launch()

## 16. File Upload

`gr.File` lets users upload any file; your function receives a file object with useful attributes like `.name`.

In [ ]:
import gradio as gr

def get_filename(file):
    return file.name

demo = gr.Interface(
    fn=get_filename,
    inputs=gr.File(label="Upload File"),
    outputs="text"
)

demo.launch()

Uploading `sample.pdf` would simply return `sample.pdf`.

## 17. Image Input

`gr.Image` hands your function a NumPy array by default, so you can inspect things like its shape.

In [ ]:
import gradio as gr

def image_info(image):
    return f"Image received: {image.shape}"

demo = gr.Interface(
    fn=image_info,
    inputs=gr.Image(),
    outputs="text"
)

demo.launch()

## 18. Dataframe

`gr.Dataframe` gives users an editable spreadsheet-like grid right in the browser.

In [ ]:
import gradio as gr

def process(data):
    return data

demo = gr.Interface(
    fn=process,
    inputs=gr.Dataframe(),
    outputs=gr.Dataframe()
)

demo.launch()

# Part 3 — Layout & Advanced Basics

## 19. Markdown

`gr.Markdown` lets you add headings, instructions, and formatted text anywhere in a `Blocks` layout.

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown(
        """
        # Student Calculator

        Enter two numbers below.
        """
    )

demo.launch()

## 20. Rows and Columns

`gr.Column` stacks components vertically; `gr.Row` places them side by side.

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    with gr.Column():
        gr.Textbox(label="First Name")
        gr.Textbox(label="Last Name")

demo.launch()

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    with gr.Row():
        gr.Textbox(label="First Name")
        gr.Textbox(label="Last Name")

demo.launch()

## 21. Tabs

Tabs help organize larger apps into separate sections the user can switch between.

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    with gr.Tab("Calculator"):
        gr.Markdown("# Calculator")

    with gr.Tab("About"):
        gr.Markdown("# About This App")

demo.launch()

## 22. State

`gr.State` holds a value in memory for the duration of a user's session — useful for anything that needs to persist between interactions, like conversation history.

In [ ]:
import gradio as gr

def add_name(name, history):
    history.append(name)

    return history, history

with gr.Blocks() as demo:

    state = gr.State([])

    name = gr.Textbox()
    button = gr.Button("Add")

    output = gr.JSON()

    button.click(
        fn=add_name,
        inputs=[name, state],
        outputs=[state, output]
    )

demo.launch()

## 23. Chatbot

For AI apps, `gr.Chatbot` is the component you'll reach for most often. A very bare-bones version can be built with a plain `Interface` first, just to see the shape of things.

In [ ]:
import gradio as gr

def chatbot(message, history):

    return f"You said: {message}"

demo = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text"
)

demo.launch()

For a proper conversational UI, Gradio offers a purpose-built component: `gr.ChatInterface`.

## 24. ChatInterface

`gr.ChatInterface` is the fastest way to get a real chat-style UI, complete with a running conversation view.

In [ ]:
import gradio as gr
import json
def chat(message, history):
    with open("history.json", "w") as f:
        json.dump(history, f, indent=2)

    return f"You said: {message}"

demo = gr.ChatInterface(
    fn=chat
)

demo.launch()

## 25. Chatbot with Simple Memory

The `history` argument automatically contains everything said so far in the conversation, so your function can use it for context.

In [ ]:
import gradio as gr

def chat(message, history):

    return f"You said: {message}\nPrevious messages: {len(history)}"

demo = gr.ChatInterface(
    fn=chat
)

demo.launch()

## 26. Build a Simple Text Summarizer

Time to combine what we've learned. This version just trims text to the first 20 words — it's **not** an LLM summarizer yet, but it demonstrates the overall Blocks + Button + Textbox pattern you'll reuse for real AI apps.

In [ ]:
import gradio as gr

def summarize(text):

    words = text.split()

    if len(words) <= 20:
        return text

    return " ".join(words[:20]) + "..."

with gr.Blocks() as demo:

    gr.Markdown("# Simple Text Summarizer")

    text = gr.Textbox(
        label="Enter Text",
        lines=10
    )

    button = gr.Button("Summarize")

    output = gr.Textbox(
        label="Summary",
        lines=5
    )

    button.click(
        fn=summarize,
        inputs=text,
        outputs=output
    )

demo.launch()

# Part 4 — AI Applications

## 27. Gradio + OpenAI

Now let's connect the chat UI to a real LLM. First, install the extra packages and store your API key in a `.env` file so it never ends up hard-coded in the notebook.

In [ ]:
!pip install openai python-dotenv

Create a file called `.env` next to this notebook with the following line:

```
OPENAI_API_KEY=your_api_key
```

In [ ]:
import os
import gradio as gr

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def chat(message, history):

    response = client.responses.create(
        model="groq/compound",
        input=message
    )

    return response.output_text

demo = gr.ChatInterface(
    fn=chat,
    title="My AI Assistant"
)

demo.launch()

The overall flow now looks like this:

```
User → Gradio UI → Python Function → OpenAI API → LLM Response → back to Gradio UI
```

## 28. Gradio + RAG

This is where everything you've learned connects to Retrieval-Augmented Generation (RAG) projects. A typical pipeline looks like:

```
PDF → PyPDF → Chunking → Embeddings → FAISS → RAG → OpenAI/Ollama → Gradio Chatbot
```

A RAG document assistant built with the components above might look like this:

```
┌──────────────────────────────────┐
│       RAG Document Assistant     │
├──────────────────────────────────┤
│ Upload PDF                       │
│ [ Choose File ]                  │
│                                  │
│ [ Process Document ]             │
├──────────────────────────────────┤
│ Ask a question                   │
│                                  │
│ ┌──────────────────────────────┐ │
│ │ What is the refund policy?   │ │
│ └──────────────────────────────┘ │
│                                  │
│ [ Ask ]                          │
├──────────────────────────────────┤
│ Answer                           │
│                                  │
│ The refund period is 30 days...  │
│                                  │
│ Source: Page 12                  │
└──────────────────────────────────┘
```

You already have every ingredient needed to build this: `gr.File` for the upload, a `Button` to trigger processing, a `Textbox` for the question, and a `Chatbot` or `Markdown` component for the answer.

## What to build next

With everything above under your belt, try building these five mini projects to cement the concepts:

1. **AI Text Summarizer** — swap the word-trimming logic in section 26 for a real LLM call
2. **Sentiment Analysis App** — classify a piece of text as positive, negative, or neutral
3. **PDF Question Answering App** — combine `gr.File` with a simple RAG pipeline
4. **AI Study Assistant** — a `ChatInterface` tuned with a system prompt for tutoring
5. **RAG Document Chatbot** — the full pipeline described in section 28, wrapped in a `Chatbot` UI

Each one reuses the same building blocks: components, events, Blocks layout, and a Python function that talks to an LLM.